In [ ]:
from langchain_core.messages import SystemMessage
!pip install langchain langchain-groq

In [ ]:
import config
import requests
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_groq import ChatGroq


In [36]:
llm = ChatGroq(
    model='llama-3.3-70b-versatile',
    api_key=config.GROQ_API_KEY
)


In [38]:

@tool("get_weather", description="Return weather info")
def get_weather(city: str):
    response = requests.get(f"https://wttr.in/{city}?format=j1")
    data = response.json()

    current = data["current_condition"][0]

    return {
        "city": city,
        "temperature_C": current["temp_C"],
        "feels_like_C": current["FeelsLikeC"],
        "humidity": current["humidity"],
        "weather": current["weatherDesc"][0]["value"]
    }


In [39]:
agent1 = create_agent(
    model=llm,
    tools=[get_weather],
    system_prompt='You are a helpful weather assistant'
)

In [44]:
inputs = {"messages": [("user", "What is the weather in Estonia")]}

for msg, meta in agent1.stream(inputs, stream_mode="messages"):
    if meta.get("langgraph_node") == "model":
        print(msg.content, end="", flush=True)


The current weather in Estonia is light rain with a temperature of 11 degrees Celsius, feeling like 9 degrees Celsius, and a humidity of 96%.